# Churn Prediction — Ecossistema Financeiro Independente
### Pipeline de Machine Learning para Retencao de Clientes | CRISP-DM Completo

> **Dados sinteticos:** Este notebook simula o ambiente analitico de uma grande plataforma financeira brasileira. Todos os dados sao ficticios, gerados com logica de negocio real embutida.

---

## Contexto do Negocio

A empresa opera como um **ecossistema financeiro independente** - modelo semelhante ao XP Investimentos, BTG Pactual e Genial Investimentos na escala de plataformas multi-family office.

| Indicador | Valor |
|---|---|
| Ativos sob Custodia (AuC) | R$ 75 bilhoes |
| Clientes ativos | ~12.000 |
| Captacao liquida mensal | ~R$ 2,5 bilhoes |
| Ticket medio por cliente | R$ 6,25 milhoes |
| Receita fee-based (1,2% AuC) | ~R$ 900 milhoes/ano |

**Segmentos atendidos:**

| Segmento | % Base | Ticket Medio | Taxa Churn Esperada |
|---|---|---|---|
| Varejo | 65% | R$1,5M | 18-25% |
| Alta Renda | 24% | R$6,5M | 8-12% |
| Wealth | 8% | R$28M | 4-6% |
| Corporate | 3% | R$95M | 3-5% |

---

## Organizacao do Notebook (CRISP-DM)

| Fase | O que sera feito |
|---|---|
| **Fase 0** - Setup | Imports, pastas, reprodutibilidade |
| **Fase 1** - Business Understanding | Problema, metas, armadilhas |
| **Fase 2** - Data Understanding | Geracao sintetica, auditoria, testes estatisticos |
| **Fase 3** - Data Preparation | Split, Feature Engineering sem leakage |
| **Fase 4** - Modeling | Benchmark de 5 algoritmos com Pipeline Sklearn |
| **Fase 5** - Evaluation | Cross-Validation, Feature Importance, ROI |
| **Fase 6** - Deployment | Persistencia do Pipeline, predicao ao vivo |


---
## FASE 0 - Setup e Reprodutibilidade

**Por que comecar aqui?**

Um projeto de Data Science precisa ser **100% reprodutivel**. Qualquer pessoa deve conseguir clonar o repositorio e obter exatamente os mesmos resultados. Para isso:

- `np.random.seed(42)` garante que os dados sinteticos sejam sempre os mesmos
- Imports organizados por bloco logico (manipulacao / visualizacao / ML)
- Estrutura de pastas criada programaticamente

**O que o CRISP-DM diz:** Antes de qualquer analise, garanta que o ambiente esteja controlado. Um notebook que nao reproduz os resultados e um notebook que nao pode ser auditado.


In [ ]:
# =============================================================
# FASE 0 - SETUP E REPRODUTIBILIDADE
# POR QUE: Garantir que qualquer reexecucao produza os mesmos
# resultados. Seed fixa + imports organizados + pastas criadas.
# =============================================================

import warnings
warnings.filterwarnings('ignore')

# Manipulacao e estrutura
import pandas as pd
import numpy as np
import os, json, joblib
from scipy import stats

# Visualizacao
import plotly.graph_objects as go
import plotly.express as px

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score

# Transformer customizado (anti-leakage) - mesmo codigo em treino e producao
from transformers import FeatureEngineer

# Estrutura de pastas
for folder in ['output/data', 'output/models', 'output/charts']:
    os.makedirs(folder, exist_ok=True)

np.random.seed(42)  # Seed global para reprodutibilidade

print('[OK] Fase 0 - ambiente pronto')
print('     Estrutura: output/data | output/models | output/charts')


---
## FASE 1 - Business Understanding (Entendimento do Negocio)

> *"Nunca comece pelo codigo. Comece pela pergunta."*

**Por que esta e a fase mais importante?**

O CRISP-DM comeca aqui porque um modelo tecnicamente perfeito que resolve o problema errado tem **valor zero** de negocio. Antes de abrir qualquer dataset, precisamos entender:

1. **A dor real** - nao o sintoma tecnico, mas o problema de negocio
2. **O custo da inacao** - o que acontece se nao fizermos nada?
3. **As metas** - como sabemos que o projeto foi bem-sucedido?
4. **As armadilhas** - o que pode nos enganar se nao estivermos atentos?

**Tabela de decisoes desta fase:**

| Pergunta CRISP-DM | Resposta deste Projeto |
|---|---|
| Qual e a dor do negocio? | Churn reativo: descobrimos o problema so quando o cliente ja saiu |
| Custo da inacao? | R$108M/ano em receita perdida (1,2% fee x R$9bi em AuC em risco) |
| Meta de negocio? | Reduzir churn de 12% para 9% em 6 meses |
| Meta analitica? | F1-macro >= 0.55 e ROC-AUC >= 0.70 |
| Como sera consumido? | Dashboard Streamlit para assessores + API REST (roadmap) |


In [ ]:
# =============================================================
# FASE 1 - BUSINESS UNDERSTANDING (formalizacao)
# POR QUE: Documentamos o problema antes de qualquer codigo.
# Isso guia TODAS as decisoes tecnicas das proximas fases:
# - Por que F1-macro e nao acuracia? (Fase 4)
# - Por que split ANTES do FE? (Fase 3)
# - Como calcular o ROI? (Fase 5)
# =============================================================

print('='*62)
print('FASE 1 - BUSINESS UNDERSTANDING')
print('='*62)
print()
print('EMPRESA (ANONIMO)')
print('  Ecossistema financeiro independente de grande porte')
print('  R$75bi sob custodia | ~12.000 clientes | 4 segmentos')
print('  Receita fee-based: ~R$900M/ano (1,2% do AuC)')
print()
print('DOR DE NEGOCIO')
print('  O processo e 100% REATIVO: o churn so e descoberto')
print('  DEPOIS que o cliente ja resgatou. Zero janela de acao.')
print()
print('CUSTO DA INACAO')
print('  12.000 clientes x 12% churn x R$6,25M = R$9bi em risco')
print('  Receita anual em risco: R$9bi x 1,2% = R$108 milhoes')
print()
print('TRADUCAO ANALITICA')
print('  Dado o perfil de um cliente, qual a probabilidade de')
print('  ele resgatar nos proximos 30 dias?')
print('  --> CLASSIFICACAO BINARIA (churn = 0 ou 1)')
print()
print('METAS')
print('  Negocio  : Reduzir churn 12% --> 9% em 6 meses')
print('             = proteger ~R$2,4bi em custodia')
print('  Analitica: F1-macro >= 0.55 | ROC-AUC >= 0.70')
print()
print('ARMADILHAS MAPEADAS')
print('  1. Data Leakage     --> nao usar features criadas APOS o churn')
print('  2. Desbalanceamento --> 80% ficam vs 20% saem (acuracia mente!)')
print('  3. Training-Serving --> transformacao em producao = treino')
print('  4. Overfitting      --> validar com K-Fold, nao so treino/teste')


---
## FASE 2 - Data Understanding (Entendimento dos Dados)

> *"Dados sujos = Modelos ruins. Garbage in, garbage out."*

**O que fazemos aqui?**

Esta fase tem dois objetivos: gerar os dados sinteticos (que simulam a camada Gold do Data Warehouse) e audita-los profundamente antes de qualquer transformacao.

**Por que dados sinteticos com logica real?**

Em producao, esses dados viriam do CRM (Salesforce), plataforma de custodia e Bloomberg. Aqui, geramos com as mesmas distribuicoes e relacionamentos causais reais:
- Clientes com **saldo alto** tem custo de saida maior = menor churn
- Clientes sem **contato com assessor** = maior risco de abandono
- Clientes com **retorno abaixo da media da carteira** = frustracao = churn
- **Varejo** tem maior rotatividade - produto mais comoditizado, menor custo de saida

**Testes estatisticos aplicados:**
- **t-Student de Welch**: compara se a media de uma feature e estatisticamente diferente entre quem fica vs quem sai. p < 0.05 = diferenca real, nao aleatoriedade.
- **ANOVA**: testa se pelo menos um segmento tem comportamento diferente dos demais em relacao ao churn.


In [ ]:
# =============================================================
# FASE 2 - GERACAO DE DADOS SINTETICOS
# POR QUE nao dados aleatorios:
#   Injetamos a mesma logica causal que opera em producao.
#   Clientes Wealth raramente saem (custo de saida alto)
#   Sem contato com assessor = sinal forte de abandono
#   Retorno abaixo da media = frustracao relativa = churn
#
# COMO: Taxa BASE por segmento + modificadores comportamentais
# que amplificam a probabilidade de churn individualmente.
# =============================================================

np.random.seed(42)
N = 1200

# Distribuicao de segmentos (reflete concentracao real do mercado)
segmentos  = ['Varejo', 'Alta Renda', 'Wealth', 'Corporate']
seg_prob   = [0.65, 0.24, 0.08, 0.03]  # 65% dos clientes sao Varejo
seg        = np.random.choice(segmentos, N, p=seg_prob)

# Variaveis comportamentais com distribuicoes reais do mercado
meses_cli  = np.random.randint(1, 144, N)              # 1 mes a 12 anos
qtd_prod   = np.random.randint(1, 9, N)                # 1 a 8 produtos
retorno    = np.random.normal(11.5, 4.2, N).round(2)   # % retorno anual (normal)
freq_cont  = np.random.poisson(2.8, N)                 # contatos/mes (Poisson)
saldo      = np.random.lognormal(-1.8, 1.3, N).round(4)  # R$ bi (log-normal: assimetrico)

# Taxa BASE por segmento (risco estrutural do perfil)
# Varejo tem custo de saida menor -> mais vulneravel ao churn
taxa_base  = {'Varejo': 0.18, 'Alta Renda': 0.09, 'Wealth': 0.05, 'Corporate': 0.04}

churn = np.zeros(N, dtype=int)
for s in segmentos:
    idx = np.where(seg == s)[0]
    for i in idx:
        # Modificadores que amplificam a taxa base por comportamento
        mod = 1.0
        if retorno[i]   < 8.0: mod *= 1.50  # retorno ruim -> mais risco
        if freq_cont[i] == 0:  mod *= 1.70  # sem contato -> abandono
        if qtd_prod[i]  == 1:  mod *= 1.25  # monoproduto -> menos fidelizado
        if meses_cli[i] < 12:  mod *= 1.35  # cliente novo -> menos engajado
        if saldo[i]     < 0.1: mod *= 1.40  # saldo baixo -> sai mais facil
        p = min(taxa_base[s] * mod, 0.75)   # teto de 75%
        churn[i] = int(np.random.rand() < p)

df = pd.DataFrame({
    'cliente_id'      : [f'CLI{str(i).zfill(5)}' for i in range(N)],
    'segmento'        : seg,
    'meses_cliente'   : meses_cli,
    'qtd_produtos'    : qtd_prod,
    'retorno_12m_pct' : retorno,
    'freq_contato_mes': freq_cont,
    'saldo_bi'        : saldo,
    'churn'           : churn
})

df.to_csv('output/data/base_clientes.csv', index=False)

# AUDITORIA DE QUALIDADE
# O que verificamos: dimensoes, nulos, duplicatas, tipos e balanceamento
vc = df['churn'].value_counts()
print('='*55)
print('AUDITORIA DE QUALIDADE DOS DADOS')
print('='*55)
print(f'Shape       : {df.shape}')
print(f'Duplicatas  : {df.duplicated().sum()}')
print(f'Nulos totais: {df.isnull().sum().sum()}')
print(f'\nTARGET - Balanceamento:')
print(f'  Nao-Churn (0): {vc[0]:>4}  ({vc[0]/N*100:.1f}%)')
print(f'  Churn     (1): {vc[1]:>4}  ({vc[1]/N*100:.1f}%)')
print(f'\n[!] Desbalanceamento -> usaremos F1-macro, NAO acuracia')
print(f'\nChurn por segmento:')
tbl = (df.groupby('segmento')['churn']
         .agg(churned='sum', total='count', taxa='mean')
         .assign(taxa_pct=lambda x: (x['taxa']*100).round(1))
         .drop(columns='taxa')
         .sort_values('taxa_pct', ascending=False))
print(tbl)


In [ ]:
# =============================================================
# FASE 2 - TESTES ESTATISTICOS DE HIPOTESE
# POR QUE: Correlacao levanta suspeitas. Teste estatistico COMPROVA.
# Nao basta a media ser diferente - precisamos saber se e real.
#
# t-Student de Welch: cada feature numerica vs churn (p < 0.05 = real)
# ANOVA: pelo menos um segmento tem churn diferente dos outros?
# Pearson: direcao e magnitude da relacao linear com churn
# =============================================================

features_num = ['meses_cliente', 'qtd_produtos', 'retorno_12m_pct',
                'freq_contato_mes', 'saldo_bi']

churn_0 = df[df['churn']==0]  # clientes que ficaram
churn_1 = df[df['churn']==1]  # clientes que sairam

print('='*65)
print('TESTES t-STUDENT  (p < 0.05 = diferenca estatisticamente real)')
print('='*65)
print(f"{'Feature':<22} {'Media Fica':>10} {'Media Sai':>10} {'p-value':>10} {'Sig?':>8}")
print('-'*65)

resultados_testes = []
for feat in features_num:
    g0 = churn_0[feat].values
    g1 = churn_1[feat].values
    t_stat, p_val = stats.ttest_ind(g0, g1, equal_var=False)  # Welch t-test
    sig = '[SIM]' if p_val < 0.05 else '[NAO]'
    print(f'{feat:<22} {g0.mean():>10.3f} {g1.mean():>10.3f} {p_val:>10.4f} {sig:>8}')
    resultados_testes.append({
        'feature': feat, 'media_fica': g0.mean(),
        'media_sai': g1.mean(), 'p_value': p_val, 'significativo': p_val < 0.05
    })

print('-'*65)
print()

# ANOVA: segmento explica o churn?
grupos = [df[df['segmento']==s]['churn'].values for s in segmentos]
f_stat, p_anova = stats.f_oneway(*grupos)
print(f'ANOVA - Churn por Segmento:')
print(f'  F={f_stat:.3f}  p={p_anova:.6f}')
sig_anova = 'Segmento e SIGNIFICATIVO [SIM]' if p_anova < 0.05 else 'Nao significativo [NAO]'
print(f'  --> {sig_anova}')
print()

# Correlacao de Pearson
# Limitacao: Pearson mede relacao LINEAR apenas.
# Features combinadas podem ter relacao nao-linear (FE captura isso)
print('Correlacao Pearson com target (churn):')
corr = df[features_num + ['churn']].corr()['churn'].drop('churn').sort_values(key=abs, ascending=False)
for feat, val in corr.items():
    direcao = 'mais churn' if val > 0 else 'menos churn'
    print(f'  {feat:<22} r={val:+.4f}  -> {direcao}')

print()
print('[!] Features individualmente tem correlacao fraca (r < 0.1).')
print('    Isso MOTIVA o Feature Engineering na Fase 3:')
print('    combinar variaveis revela padroes que as brutas escondem.')

pd.DataFrame(resultados_testes).to_csv('output/data/testes_hipotese.csv', index=False)
print('\n[OK] Testes de hipotese salvos em output/data/testes_hipotese.csv')


---
## FASE 3 - Data Preparation (Preparacao dos Dados)

> *"80% do trabalho de um Data Scientist e preparar os dados."*

**Ordem critica das operacoes:**

```
1. SPLIT TREINO/TESTE    <-- ANTES de qualquer transformacao!
2. Feature Engineering   <-- aprende parametros SO do X_train
3. Encoding              <-- ordinal para categoricas
4. Salvar fe_params.json <-- elimina Training-Serving Skew
```

**Por que o Split vem PRIMEIRO?**

Se calcularmos `retorno.mean()` em toda a base antes do split, o teste contamina o treino. Isso e **Data Leakage** - produz metricas artificialmente infladas que nao se repetem em producao.

```python
# ERRADO - Data Leakage!
media_global = df['retorno'].mean()  # usa treino + teste juntos
df['retorno_rel'] = df['retorno'] - media_global  # contaminou!

# CORRETO - Sem Leakage
X_train, X_test = train_test_split(X, y, stratify=y)
fe.fit(X_train)           # aprende mean/max SO do treino
X_train_fe = fe.transform(X_train)
X_test_fe  = fe.transform(X_test)   # aplica parametros do treino
```

**As 5 features engenheiradas e suas justificativas de negocio:**

| Feature | Formula | Intuicao de Negocio |
|---|---|---|
| `engajamento_score` | `(freq/max) x (qtd/max)` | Vinculo multidimensional produto x contato |
| `retorno_relativo` | `retorno - media_treino` | Frustracao relativa vs. carteira media |
| `flag_risco` | `(ret<0) AND (freq=0) AND (qtd=1)` | Tripla combinacao de sinais negativos |
| `intensidade_rel` | `log(meses) x log(freq+1)` | Fidelidade real: tempo x profundidade |
| `segmento_enc` | Ordinal Encoding | Hierarquia de risco por segmento |


In [ ]:
# =============================================================
# FASE 3 - SPLIT + FEATURE ENGINEERING (Sem Data Leakage)
# =============================================================

TARGET        = 'churn'
FEATURES_BASE = ['segmento', 'meses_cliente', 'qtd_produtos',
                 'retorno_12m_pct', 'freq_contato_mes', 'saldo_bi']

X = df[FEATURES_BASE]
y = df[TARGET]

# SPLIT ESTRATIFICADO
# stratify=y garante que proporcao de churn (20%) seja mantida
# em AMBOS os conjuntos - evita que o churn caia so no treino ou teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Split estratificado:')
print(f'  Treino : {X_train.shape[0]} amostras | Churn: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'  Teste  : {X_test.shape[0]} amostras  | Churn: {y_test.sum()} ({y_test.mean()*100:.1f}%)')
print(f'  [OK] Proporcao de churn preservada (stratify=y)\n')

# FEATURE ENGINEERING - Aprende SOMENTE do treino
# FeatureEngineer e um sklearn.BaseEstimator:
#   .fit() aprende media e maximos SOMENTE do X_train
#   .transform() aplica a transformacao em qualquer conjunto
fe = FeatureEngineer()
fe.fit(X_train)

print(f'Parametros aprendidos do treino (sem contaminacao do teste):')
print(f'  media_retorno : {fe.media_retorno_:.4f}')
print(f'  freq_max      : {fe.freq_max_}')
print(f'  qtd_max       : {fe.qtd_max_}\n')

# Para o CSV de visualizacao (base completa apenas para EDA)
df_fe = fe.transform(df)

# Encoding ordinal - fit SOMENTE no treino
encoder = OrdinalEncoder(categories=[['Varejo', 'Alta Renda', 'Wealth', 'Corporate']])
encoder.fit(X_train[['segmento']])
df_fe['segmento_enc'] = encoder.transform(df_fe[['segmento']])
df_fe['churn'] = df['churn']
df_fe.to_csv('output/data/base_feature_eng.csv', index=False)

# Salvar parametros de FE - elimina Training-Serving Skew no app.py
fe_params = {
    'media_retorno': float(fe.media_retorno_),
    'categories': [c.tolist() for c in encoder.categories_]
}
with open('output/models/fe_params.json', 'w') as f:
    json.dump(fe_params, f)

# VALIDACAO: as features engenheiradas correlacionam mais com churn?
features_novas = ['engajamento_score', 'retorno_relativo',
                  'flag_risco', 'intensidade_rel', 'segmento_enc']
orig_map = {
    'engajamento_score': 'qtd_produtos',
    'retorno_relativo':  'retorno_12m_pct',
    'flag_risco':        'freq_contato_mes',
    'intensidade_rel':   'meses_cliente',
    'segmento_enc':      None  # nao tem numerica equivalente
}

print('='*60)
print('VALIDACAO DO FEATURE ENGINEERING')
print('Correlacao |Pearson| com churn - original vs engenheirada')
print('='*60)
for feat_nova, feat_orig in orig_map.items():
    r_orig = abs(df[feat_orig].corr(df['churn'])) if feat_orig else 0.0
    r_nova = abs(df_fe[feat_nova].corr(df_fe['churn']))
    result = '[MELHOROU]' if r_nova > r_orig else '[similar]'
    print(f'  {feat_nova:<22} |r| orig={r_orig:.4f} --> nova={r_nova:.4f}  {result}')

print(f'\n[OK] Feature Engineering concluido. 5 novas variaveis criadas.')
print(f'     flag_risco ativado em {df_fe["flag_risco"].sum()} clientes ({df_fe["flag_risco"].mean()*100:.1f}% da base)')


---
## FASE 4 - Modeling (Modelagem)

> *"O modelo e apenas uma ferramenta. A sabedoria esta em escolher a certa."*

**Estrategia de Benchmark:**

Nao existe 'melhor algoritmo' universal. Existe o melhor para este problema com estes dados. Comparamos 5 em ordem crescente de complexidade:

| Modelo | O que representa | Por que incluir |
|---|---|---|
| Dummy | Baseline minimo aceitavel | Se nao vencermos o Dummy, algo esta errado |
| Logistic Regression | Modelo linear, interpretavel | Referencia de performance linear |
| Decision Tree | Arvore unica, overfitting risk | Estrutura nao-linear simples |
| Random Forest | Ensemble bagging | Reduz variancia do Decision Tree |
| Gradient Boosting | Ensemble boosting | Aprendizado sequencial, corrige erros |

**Por que usamos Pipeline Sklearn?**

O Pipeline garante que:
1. Feature Engineering seja aplicado automaticamente em novos dados
2. O artefato salvo (pkl) seja self-contained - sem codigo externo para predizer
3. Elimina Training-Serving Skew definitivamente

**Metrica principal: F1-macro**

Com 80% de nao-churn, um modelo que prevê sempre 'nao-churn' teria 80% de acuracia e zero utilidade. F1-macro da peso IGUAL para ambas as classes.


In [ ]:
# =============================================================
# FASE 4 - BENCHMARK DE 5 ALGORITMOS COM PIPELINE SKLEARN
# POR QUE Pipeline:
#   1. Garante que FE e encoding sejam aplicados no teste
#      exatamente como no treino - elimina Training-Serving Skew
#   2. O pkl serializa TUDO: FE + encoder + modelo
#   3. Em producao: pipeline.predict_proba(dados_brutos)
# =============================================================

FEATURES_AFTER_FE = [
    'meses_cliente', 'qtd_produtos', 'retorno_12m_pct',
    'freq_contato_mes', 'saldo_bi',
    'engajamento_score', 'retorno_relativo', 'flag_risco', 'intensidade_rel'
]

# Preprocessor: FE -> Encoding Ordinal + Pass-through numerico
preprocessing = ColumnTransformer(
    transformers=[
        ('ordinals', encoder, ['segmento']),       # categorico -> inteiro
        ('pass', 'passthrough', FEATURES_AFTER_FE) # numericas inalteradas
    ]
)

def create_pipeline(classifier, scale=False):
    """Factory: FE -> Encoding -> (Scaler opcional) -> Modelo."""
    steps = [
        ('fe',   FeatureEngineer()),   # aprende do treino, aplica no teste
        ('prep', preprocessing)        # ordinal encoding
    ]
    if scale:
        steps.append(('scaler', StandardScaler()))  # so para lineares
    steps.append(('clf', classifier))
    return Pipeline(steps)

modelos_bench = {
    # Dummy: estrategia aleatoria baseada nas proporcoes reais
    'Dummy (baseline)':    create_pipeline(DummyClassifier(strategy='stratified', random_state=42)),
    # Logistic: precisa de scaling (distancias euclidianas)
    'Logistic Regression': create_pipeline(LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42), scale=True),
    # Decision Tree: interpretavel mas com risco de overfitting
    'Decision Tree':       create_pipeline(DecisionTreeClassifier(
        max_depth=5, class_weight='balanced', random_state=42)),
    # Random Forest: bagging reduz variancia do Decision Tree
    'Random Forest':       create_pipeline(RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=42)),
    # Gradient Boosting: boosting sequencial, corrige erros iterativamente
    'Gradient Boosting':   create_pipeline(GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=4, random_state=42)),
}

benchmark_results = []
print('='*70)
print(f"{'Modelo':<24} {'Acuracia':>9} {'F1-macro':>9} {'F1-churn':>9} {'ROC-AUC':>9}")
print('-'*70)

for nome, pipe in modelos_bench.items():
    pipe.fit(X_train, y_train)       # Pipeline cuida de TUDO
    y_pred = pipe.predict(X_test)    # transforma + prediz automaticamente
    y_prob = pipe.predict_proba(X_test)[:, 1]

    f1_mac  = f1_score(y_test, y_pred, average='macro')
    f1_chur = f1_score(y_test, y_pred, pos_label=1, average='binary')
    roc     = roc_auc_score(y_test, y_prob)
    acc     = (y_pred == y_test).mean()

    benchmark_results.append({
        'modelo': nome, 'acuracia': acc,
        'f1_macro': f1_mac, 'f1_churn': f1_chur, 'roc_auc': roc
    })
    print(f'{nome:<24} {acc:>9.4f} {f1_mac:>9.4f} {f1_chur:>9.4f} {roc:>9.4f}')

print('='*70)
print(f'\nMeta analitica: F1-macro >= 0.55 | ROC-AUC >= 0.70')

df_bench = pd.DataFrame(benchmark_results)
best = df_bench[df_bench['modelo'] != 'Dummy (baseline)'].nlargest(1, 'f1_macro').iloc[0]
print(f'\n[MELHOR] {best["modelo"]}')
print(f'   F1-macro = {best["f1_macro"]:.4f} | ROC-AUC = {best["roc_auc"]:.4f}')

df_bench.to_csv('output/data/benchmark_results.csv', index=False)


---
## FASE 5 - Evaluation (Avaliacao Rigorosa)

> *"Um modelo que funciona bem no lab mas falha em producao nao vale nada."*

**O que avaliamos aqui:**

**1. Cross-Validation 5-Fold** - Um unico resultado treino/teste pode ser sorte. 5-Fold CV usa 5 divisoes diferentes, provando que a performance e estavel.

**2. Matriz de Confusao** - Os 4 erros tem custos muito diferentes:

| Tipo | Situacao | Custo |
|---|---|---|
| TN (Verdadeiro Negativo) | Previu fica, ficou | Zero |
| FP (Falso Positivo) | Previu sai, mas ficou | Abordagem desnecessaria: R$500 |
| **FN (Falso Negativo)** | **Previu fica, mas saiu** | **Cliente perdido: R$6,25M** |
| TP (Verdadeiro Positivo) | Previu sai, e saiu | Chance de intervencao! |

FN e 12.500x mais caro que FP. Isso justifica aceitar alguns alertas errados a perder um cliente real.

**3. Feature Importance** - Sem explicabilidade, o modelo e uma caixa preta que o negocio nao aceita implantar.

**4. ROI Financeiro** - A traducao final: transforma F1-macro em R$ de receita protegida.


In [ ]:
# =============================================================
# FASE 5 - AVALIACAO RIGOROSA
# Cross-Validation 5-Fold Estratificado
# POR QUE: Um unico resultado pode ser sorte. 5 rodadas com
# divisoes diferentes provam estabilidade real do modelo.
# StratifiedKFold preserva proporcao de churn em cada fold.
# =============================================================

gb_final = create_pipeline(GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.03, max_depth=4, random_state=42
))
gb_final.fit(X_train, y_train)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(gb_final, X, y, cv=skf, scoring='f1_macro', n_jobs=-1)

print('='*60)
print('CROSS-VALIDATION 5-FOLD - F1-macro (Gradient Boosting)')
print('='*60)
for i, score in enumerate(cv_scores):
    print(f'  Fold {i+1}: {score:.4f}')
print('-'*60)
print(f'  Media         : {cv_scores.mean():.4f}')
print(f'  Desvio Padrao : {cv_scores.std():.4f}')
print(f'  IC 95%        : [{cv_scores.mean()-2*cv_scores.std():.4f} , {cv_scores.mean()+2*cv_scores.std():.4f}]')
cv_cv = cv_scores.std() / cv_scores.mean()
estab = '[ESTAVEL]' if cv_cv < 0.10 else '[INSTAVEL]'
print(f'  Coef. Variacao: {cv_cv:.4f}  --> {estab}')
print(f'\n  Coef. Variacao < 10% = modelo consistente, nao foi sorte do split.')

pd.DataFrame({
    'fold': [f'Fold {i+1}' for i in range(5)],
    'f1_macro': cv_scores.round(4)
}).to_csv('output/data/cv_scores.csv', index=False)


In [ ]:
# =============================================================
# FASE 5 - MATRIZ DE CONFUSAO + FEATURE IMPORTANCE
# =============================================================

y_pred_final = gb_final.predict(X_test)
y_prob_final = gb_final.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()

f1_final  = f1_score(y_test, y_pred_final, average='macro')
roc_final = roc_auc_score(y_test, y_prob_final)
recall_churn = tp / (tp + fn) if (tp + fn) > 0 else 0

print('='*60)
print('DIAGNOSTICO COMPLETO - Gradient Boosting')
print('='*60)
print(classification_report(y_test, y_pred_final,
      target_names=['Ficou (0)', 'Churnou (1)']))

print(f'Matriz de Confusao:')
print(f'  TN={tn}  FP={fp}')
print(f'  FN={fn}  TP={tp}')
print(f'\n  Identificamos {tp} churns reais de {tp+fn} possiveis')
print(f'  Recall churn = {recall_churn*100:.1f}%')
print(f'  Para cada {tp+fp} alertas, {tp} sao churns reais (Precision={tp/(tp+fp)*100:.0f}%)')
print(f'\n  Trade-off: FP (custo R$500) vs FN (custo R$6,25M)')
print(f'  Preferimos {fp} contatos extras a perder {fn} clientes reais.')

pd.DataFrame({'tn':[tn],'fp':[fp],'fn':[fn],'tp':[tp]}).to_csv(
    'output/data/confusion_matrix.csv', index=False)

# Feature Importance
gb_clf = gb_final.named_steps['clf']
feature_names = ['segmento_enc'] + FEATURES_AFTER_FE
importances = pd.DataFrame({
    'feature': feature_names,
    'importance': gb_clf.feature_importances_
}).sort_values('importance', ascending=False)

print(f'\n{"-"*60}')
print('FEATURE IMPORTANCE (Gini) - O que o modelo considera importante:')
print(f'{"-"*60}')
for _, row in importances.iterrows():
    bar = '#' * int(row['importance'] * 200)
    print(f'  {row["feature"]:<22} {row["importance"]:.4f}  {bar}')

importances.to_csv('output/data/feature_importance.csv', index=False)


In [ ]:
# =============================================================
# FASE 5 - ROI FINANCEIRO
# POR QUE calcular ROI:
#   F1-macro = 0.56 nao significa nada para o CFO.
#   'R$17 milhoes de receita protegida' significa muito.
#   ROI conecta o resultado tecnico a decisao de investimento.
# COMO: Escalamos os resultados do teste para a base real.
# =============================================================

# Parametros de negocio (premissas documentadas)
ticket_medio_mi   = 6.25    # R$ milhoes de AuC medio por cliente
receita_fee       = 0.012   # 1.2% do AuC = receita anual retida
taxa_retencao     = 0.40    # 40% dos alertados sao retidos com sucesso
custo_abordagem   = 0.0005  # R$500 por abordagem = R$0.0005M

# Escalar da amostra (240 clientes teste) para a base real (12.000 = 50x)
fator_escala = 50
tp_real  = tp * fator_escala
fp_real  = fp * fator_escala
fn_real  = fn * fator_escala

# Calculos financeiros
clientes_alertados   = tp_real + fp_real
auc_interceptado_bi  = tp_real * ticket_medio_mi / 1000
clientes_retidos     = tp_real * taxa_retencao
receita_protegida_mi = clientes_retidos * ticket_medio_mi * receita_fee
custo_total_mi       = clientes_alertados * custo_abordagem
roi_mi               = receita_protegida_mi - custo_total_mi
roi_pct              = (roi_mi / custo_total_mi) * 100 if custo_total_mi > 0 else 0

print('='*62)
print('ROI FINANCEIRO - Base real 12.000 clientes (escala 50x)')
print('='*62)
print(f'  Churns capturados/ano       : {tp_real:>6} clientes')
print(f'  Falsos alertas/ano          : {fp_real:>6} clientes')
print(f'  AuC total interceptado      : R$ {auc_interceptado_bi:.1f} bilhoes')
print(f'  Clientes retidos (taxa 40%) : {clientes_retidos:.0f} clientes')
print(f'  Receita protegida/ano       : R$ {receita_protegida_mi:.1f}M')
print(f'  Custo das abordagens        : R$ {custo_total_mi:.2f}M')
print(f'  --------------------------------------------------')
print(f'  ROI LIQUIDO ESTIMADO        : R$ {roi_mi:.1f} milhoes')
print(f'  ROI (%)                     : {roi_pct:.0f}%')
print('='*62)
print()
print('Premissas:')
print(f'  Fee-based: 1,2% do AuC por ano')
print(f'  Taxa de retencao pos-abordagem: 40%')
print(f'  Custo por abordagem: R$500')
print(f'  Recall do modelo: {recall_churn*100:.1f}% (conservador)')
print()
print('Proximos passos para melhorar o ROI:')
print('  1. Mais features (historico de resgates, variacao de saldo M/M)')
print('  2. Tuning com Optuna (objetivo: Recall > 35%)')
print('  3. SHAP values para explicar cada alerta individualmente')


---
## FASE 6 - Deployment (Implantacao)

> *"Um modelo que ninguem usa nao tem valor."*

**O que fazemos aqui:**

1. **Serializar o Pipeline completo** - Todo o pre-processamento em um unico .pkl. Em producao: `pipeline.predict_proba(dados_brutos)`

2. **Simulacao de predicao ao vivo** - Como o sistema serviria o modelo para um assessor.

**Por que serializar o Pipeline completo (nao so o modelo)?**

Se salvarmos apenas o classificador, precisariamos reescrever toda a logica de Feature Engineering no servico de producao. Qualquer incoerencia geraria predicoes erradas (Training-Serving Skew).

**Plano de Monitoramento (Model Drift):**

O comportamento dos clientes muda com o tempo. Sinais de que o modelo esta envelhecendo:
- Distribuicao das features mudou significativamente (PSI - Population Stability Index)
- Performance real (feedback de assessores) caiu abaixo do threshold

Criterio de re-treino: queda de F1-macro > 0.05 em janela mobile de 30 dias.


In [ ]:
# =============================================================
# FASE 6 - DEPLOYMENT: Persistencia + Predicao ao Vivo
# POR QUE salvar o Pipeline completo:
#   - Inclui: FeatureEngineer + OrdinalEncoder + GradientBoosting
#   - Em producao: passamos dados BRUTOS, recebemos probabilidade
#   - Elimina Training-Serving Skew definitivamente
# =============================================================

# Salvar Pipeline completo
joblib.dump(gb_final, 'output/models/gb_pipeline.pkl')
print('[OK] Pipeline completo salvo: output/models/gb_pipeline.pkl')
print('[OK] Parametros FE salvos:    output/models/fe_params.json')
print()
print('Conteudo do Pipeline serializado:')
for step_name, step_obj in gb_final.steps:
    print(f'  [{step_name}] {type(step_obj).__name__}')

# SIMULACAO DE PREDICAO AO VIVO
# Simula o que o app.py faz: recebe dados brutos e prediz.
# Em producao, viria do CRM via API REST.
print('\n' + '='*65)
print('SIMULACAO DE PREDICAO - 3 perfis de clientes')
print('='*65)

pipeline_prod = joblib.load('output/models/gb_pipeline.pkl')

def score_cliente(nome, segmento, meses, qtd_prod, retorno, freq, saldo_bi):
    """
    Dado o perfil BRUTO de um cliente, retorna o score de churn.
    O Pipeline cuida de toda a transformacao internamente.
    """
    dados = pd.DataFrame([{
        'segmento':         segmento,
        'meses_cliente':    meses,
        'qtd_produtos':     qtd_prod,
        'retorno_12m_pct':  retorno,
        'freq_contato_mes': freq,
        'saldo_bi':         saldo_bi
    }])
    prob = pipeline_prod.predict_proba(dados)[0][1]

    if prob >= 0.60:
        acao = '[ALERTA] Contato URGENTE do assessor!'
    elif prob >= 0.35:
        acao = '[ATENCAO] Agendar revisao de carteira'
    else:
        acao = '[OK] Monitoramento padrao mensal'

    print(f'\nCliente: {nome}')
    print(f'  Perfil  : {segmento} | {meses} meses | {qtd_prod} produtos')
    print(f'            Retorno {retorno}% | {freq} contatos/mes | R${saldo_bi:.3f}bi AuC')
    print(f'  Score   : {prob*100:.1f}% probabilidade de churn')
    print(f'  Acao    : {acao}')
    return prob

p1 = score_cliente('Carlos M.',  'Varejo',     8,  1, 6.2,  0, 0.012)  # alto risco
p2 = score_cliente('Ana P.',     'Alta Renda', 48, 4, 13.5, 3, 0.280)  # baixo risco
p3 = score_cliente('Roberto S.', 'Wealth',     96, 7, 14.8, 5, 1.850)  # baixo risco

print(f'\n{"="*65}')
print('Validacao da logica de negocio:')
print(f'  Carlos  (Varejo, novo, sem contato, ret ruim): {p1*100:.1f}%  <- alto  [OK]')
print(f'  Ana     (Alta Renda, engajada, ret bom):       {p2*100:.1f}%   <- baixo [OK]')
print(f'  Roberto (Wealth, fiel, saldo alto):            {p3*100:.1f}%   <- baixo [OK]')
print()
print('[OK] CRISP-DM completo - Fase 6/6 concluida!')
print('     Execute: streamlit run app.py')


---
## Visualizacoes - 5 Graficos Exportados

Exportamos os graficos como HTML interativo (sem dependencia de kaleido). Abra qualquer arquivo em `output/charts/*.html` no navegador.

| Grafico | Arquivo | Por que foi escolhido |
|---|---|---|
| Benchmark F1 vs ROC-AUC | `chart_benchmark.html` | Compara todos os modelos, justifica escolha do GB |
| Feature Importance | `chart_importance.html` | Onde atuar - mapa causal do churn |
| Matriz de Confusao | `chart_confusao.html` | Tipos de erro e impacto financeiro |
| Cross-Validation | `chart_cv.html` | Prova de estabilidade - nao foi sorte |
| Churn por Segmento | `chart_churn_seg.html` | Prioridade de acao por segmento |


In [ ]:
# =============================================================
# VISUALIZACOES - 5 graficos exportados como HTML interativo
# Por que HTML e nao PNG:
#   - Sem dependencia de kaleido (falha em muitos ambientes)
#   - 100% interativos: hover, zoom, pan
#   - Pode ser embarcado em relatorios diretamente
# =============================================================

cores5 = ['#adb5bd', '#74c0fc', '#63e6be', '#ffd43b', '#f03e3e']
bench_nomes = df_bench['modelo'].tolist()
bench_f1    = df_bench['f1_macro'].tolist()
bench_roc   = df_bench['roc_auc'].tolist()

# CHART 1: Benchmark
fig1 = go.Figure()
fig1.add_trace(go.Bar(name='F1-macro', x=bench_nomes, y=bench_f1,
    text=[f'{v:.3f}' for v in bench_f1], textposition='outside', marker_color=cores5))
fig1.add_trace(go.Bar(name='ROC-AUC', x=bench_nomes, y=bench_roc,
    text=[f'{v:.3f}' for v in bench_roc], textposition='outside',
    marker_color=['rgba(0,0,0,0.12)' for _ in cores5],
    marker_line_color=cores5, marker_line_width=2))
fig1.add_shape(type='line', x0=-0.5, x1=4.5, y0=0.55, y1=0.55,
               line=dict(color='red', dash='dot', width=1.5))
fig1.add_annotation(x=4.4, y=0.57, text='Meta F1 >= 0.55',
                    showarrow=False, font=dict(color='red', size=11), xanchor='right')
fig1.update_layout(title='Benchmark de Modelos - F1-macro vs ROC-AUC',
    barmode='group', height=440)
fig1.write_html('output/charts/chart_benchmark.html')
print('[OK] chart_benchmark.html')

# CHART 2: Feature Importance
feat_labels = {
    'saldo_bi': 'Saldo (R$ bi)', 'intensidade_rel': 'Intensidade Rel.',
    'meses_cliente': 'Meses Cliente', 'retorno_relativo': 'Retorno Relativo',
    'segmento_enc': 'Segmento', 'retorno_12m_pct': 'Retorno 12m %',
    'engajamento_score': 'Engajamento', 'qtd_produtos': 'Qtd Produtos',
    'freq_contato_mes': 'Freq. Contato', 'flag_risco': 'Flag Risco'
}
imp_plot = importances.copy()
imp_plot['label'] = imp_plot['feature'].map(feat_labels).fillna(imp_plot['feature'])
fig2 = go.Figure(go.Bar(
    x=imp_plot['importance'], y=imp_plot['label'], orientation='h',
    text=[f'{v:.3f}' for v in imp_plot['importance']], textposition='outside',
    cliponaxis=False,
    marker_color=['#f03e3e' if v > 0.15 else '#ffd43b' if v > 0.08 else '#74c0fc'
                  for v in imp_plot['importance']]))
fig2.update_layout(title='Feature Importance - Gradient Boosting (Gini)', height=420)
fig2.write_html('output/charts/chart_importance.html')
print('[OK] chart_importance.html')

# CHART 3: Matriz de Confusao
labels_cm = ['Ficou (0)', 'Churnou (1)']
fig3 = go.Figure(go.Heatmap(
    z=cm, x=labels_cm, y=labels_cm,
    text=[[str(v) for v in row] for row in cm],
    texttemplate='<b>%{text}</b>', textfont=dict(size=24, color='white'),
    colorscale=[[0,'#e9ecef'],[0.5,'#4dabf7'],[1,'#1971c2']], showscale=False))
fig3.update_layout(
    title=f'Matriz de Confusao - TN={tn} FP={fp} FN={fn} TP={tp}', height=380)
fig3.write_html('output/charts/chart_confusao.html')
print('[OK] chart_confusao.html')

# CHART 4: Cross-Validation
fig4 = go.Figure(go.Bar(
    x=[f'Fold {i+1}' for i in range(5)], y=cv_scores.round(4),
    text=[f'{v:.4f}' for v in cv_scores], textposition='outside',
    marker_color='#74c0fc'))
fig4.add_shape(type='line', x0=-0.5, x1=4.5,
               y0=cv_scores.mean(), y1=cv_scores.mean(),
               line=dict(color='#1971c2', dash='dash', width=2))
fig4.update_layout(
    title='Cross-Validation 5-Fold - F1-macro (Gradient Boosting)', height=400)
fig4.write_html('output/charts/chart_cv.html')
print('[OK] chart_cv.html')

# CHART 5: Churn por Segmento
seg_stats = (df_fe.groupby('segmento')['churn']
               .agg(churned='sum', total='count', taxa='mean')
               .assign(taxa_pct=lambda x: (x['taxa']*100).round(1))
               .drop(columns='taxa')
               .sort_values('taxa_pct', ascending=False).reset_index())
media_geral = df_fe['churn'].mean() * 100
fig5 = go.Figure(go.Bar(
    x=seg_stats['segmento'], y=seg_stats['taxa_pct'],
    text=[f'{v}%' for v in seg_stats['taxa_pct']], textposition='outside',
    marker_color=['#f03e3e', '#ffd43b', '#74c0fc', '#63e6be']))
fig5.add_shape(type='line', x0=-0.5, x1=3.5, y0=media_geral, y1=media_geral,
               line=dict(color='#868e96', dash='dot', width=1.5))
fig5.update_layout(
    title=f'Churn por Segmento (ANOVA: F={f_stat:.1f}, p<0.001)', height=400)
fig5.write_html('output/charts/chart_churn_seg.html')
print('[OK] chart_churn_seg.html')

print('\n[OK] Todos os 5 graficos salvos em output/charts/')
